# AgentGPT-Style Multi-Agent Business Intelligence Platform

Educational prototype coordinating four specialist Python agents to analyze business CSV data: Data Analyst, KPI Analyst, Insight Agent, and Executive Reporter. The default data is synthetic sales data. No LLM API key is needed. This is a deterministic starter workflow, not a production autonomous-agent system.

## 1. Install dependencies

In [ ]:
%pip install -q pandas numpy matplotlib

## 2. Imports

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dataclasses import dataclass, asdict
from typing import Any, Dict, List

np.random.seed(42)
pd.set_option("display.max_columns", 50)

## 3. Load or generate business data

In [ ]:
CSV_PATH = None  # Set to a CSV path to use your own data.

def make_demo_data(n=500, seed=42):
    rng = np.random.default_rng(seed)
    dates = pd.date_range("2025-01-01", periods=365, freq="D")
    df = pd.DataFrame({
        "order_id": [f"ORD-{i:05d}" for i in range(1, n+1)],
        "order_date": rng.choice(dates, n),
        "region": rng.choice(["North", "South", "East", "West"], n),
        "channel": rng.choice(["Online", "Retail", "Partner"], n),
        "category": rng.choice(["Electronics", "Home", "Office", "Accessories"], n),
        "units": rng.integers(1, 8, n),
        "unit_price": np.round(rng.uniform(15, 350, n), 2),
        "discount_pct": rng.choice([0, .05, .10, .15], n),
        "cost_per_unit": np.round(rng.uniform(8, 250, n), 2),
    })
    df["revenue"] = (df["units"] * df["unit_price"] * (1-df["discount_pct"])).round(2)
    df["cost"] = (df["units"] * df["cost_per_unit"]).round(2)
    df["profit"] = (df["revenue"] - df["cost"]).round(2)
    return df.sort_values("order_date").reset_index(drop=True)

data = pd.read_csv(CSV_PATH) if CSV_PATH else make_demo_data()
if "order_date" in data.columns:
    data["order_date"] = pd.to_datetime(data["order_date"], errors="coerce")
print("Rows:", len(data), "Columns:", len(data.columns))
display(data.head())

## 4. Data quality validation

In [ ]:
def validate_data(df):
    return {
        "rows": int(len(df)),
        "columns": int(len(df.columns)),
        "duplicate_rows": int(df.duplicated().sum()),
        "missing_by_column": df.isna().sum().astype(int).to_dict(),
        "numeric_columns": df.select_dtypes(include=np.number).columns.tolist(),
    }

quality = validate_data(data)
print(json.dumps(quality, indent=2, default=str))
if "revenue" not in data.columns:
    raise ValueError("CSV must include a numeric 'revenue' column for the baseline KPI workflow.")

## 5. Agent result structure

In [ ]:
@dataclass
class AgentResult:
    agent: str
    status: str
    summary: str
    findings: List[str]
    metrics: Dict[str, Any]

def to_dict(result):
    return asdict(result)

## 6. Data Analyst Agent

In [ ]:
def data_analyst_agent(df):
    numeric = df.select_dtypes(include=np.number)
    findings = [
        f"Dataset contains {len(df):,} rows and {len(df.columns)} columns.",
        f"{int(df.duplicated().sum())} duplicate rows detected.",
        f"{int(df.isna().sum().sum())} missing cells detected."
    ]
    metrics = {
        "numeric_summary": numeric.describe().round(2).to_dict(),
        "missing_by_column": df.isna().sum().astype(int).to_dict(),
        "duplicate_rows": int(df.duplicated().sum()),
    }
    return AgentResult("Data Analyst Agent", "ok", "Profiled data structure and quality.", findings, metrics)

profile = data_analyst_agent(data)
for item in profile.findings: print("-", item)

## 7. KPI Analyst Agent

In [ ]:
def kpi_analyst_agent(df):
    metrics, findings = {}, []
    for col in ["revenue", "cost", "profit", "units"]:
        if col in df.columns:
            metrics[f"total_{col}"] = round(float(df[col].fillna(0).sum()), 2)
    metrics["average_order_revenue"] = round(float(df["revenue"].mean()), 2)
    if "profit" in df.columns:
        total_revenue = float(df["revenue"].fillna(0).sum())
        total_profit = float(df["profit"].fillna(0).sum())
        metrics["profit_margin_pct"] = round(total_profit / total_revenue * 100, 2) if total_revenue else None
        findings.append(f"Profit margin: {metrics['profit_margin_pct']}%." if metrics["profit_margin_pct"] is not None else "Profit margin unavailable because revenue sums to zero.")
    if "order_id" in df.columns:
        metrics["unique_orders"] = int(df["order_id"].nunique())
    findings.append(f"Calculated {len(metrics)} available KPI values.")
    return AgentResult("KPI Analyst Agent", "ok", "Calculated available business KPIs.", findings, metrics)

kpis = kpi_analyst_agent(data)
print(json.dumps(kpis.metrics, indent=2))

## 8. Insight Agent

In [ ]:
def insight_agent(df):
    findings, metrics = [], {}
    if "region" in df.columns:
        by_region = df.groupby("region", dropna=False)["revenue"].sum().sort_values(ascending=False)
        metrics["revenue_by_region"] = by_region.round(2).to_dict()
        if len(by_region): findings.append(f"Top region by revenue: {by_region.index[0]} ({by_region.iloc[0]:,.2f}).")
    if "category" in df.columns:
        by_category = df.groupby("category", dropna=False)["revenue"].sum().sort_values(ascending=False)
        metrics["revenue_by_category"] = by_category.round(2).to_dict()
        if len(by_category): findings.append(f"Top category by revenue: {by_category.index[0]} ({by_category.iloc[0]:,.2f}).")
    if "order_date" in df.columns:
        temp = df.dropna(subset=["order_date"]).copy()
        if not temp.empty:
            temp["month"] = temp["order_date"].dt.to_period("M").astype(str)
            monthly = temp.groupby("month")["revenue"].sum().sort_index()
            metrics["monthly_revenue"] = monthly.round(2).to_dict()
            if len(monthly) >= 2 and monthly.iloc[0] != 0:
                pct = (monthly.iloc[-1] - monthly.iloc[0]) / abs(monthly.iloc[0]) * 100
                metrics["first_to_last_month_change_pct"] = round(float(pct), 2)
                findings.append(f"Revenue change from first to last observed month: {pct:.2f}%.")
    if not findings: findings.append("No compatible dimensions available for segment or time insights.")
    return AgentResult("Insight Agent", "ok", "Generated deterministic segment and trend insights.", findings, metrics)

insights = insight_agent(data)
for item in insights.findings: print("-", item)

## 9. Visualize revenue by region

In [ ]:
if "region" in data.columns:
    data.groupby("region")["revenue"].sum().sort_values(ascending=False).plot(
        kind="bar", title="Revenue by Region", xlabel="Region", ylabel="Revenue"
    )
    plt.tight_layout()
    plt.show()
else:
    print("No region column available.")

## 10. Executive Reporter Agent

In [ ]:
def executive_reporter_agent(agent_results):
    findings, metrics = [], {}
    for result in agent_results:
        findings.extend([f"[{result.agent}] {f}" for f in result.findings])
        metrics[result.agent] = result.metrics
    return AgentResult(
        "Executive Reporter Agent", "ok",
        "BI workflow completed. Validate data quality and metric definitions before making decisions.",
        findings, metrics
    )

report = executive_reporter_agent([profile, kpis, insights])
print(report.summary)
for item in report.findings: print("-", item)

## 11. Export JSON and Markdown report

In [ ]:
report_dict = to_dict(report)
with open("business_intelligence_report.json", "w", encoding="utf-8") as f:
    json.dump(report_dict, f, indent=2, default=str)

lines = ["# Business Intelligence Report", "", report.summary, "", "## Findings"]
lines += [f"- {item}" for item in report.findings]
lines += ["", "## Metrics", "```json", json.dumps(report.metrics, indent=2, default=str), "```"]
with open("business_intelligence_report.md", "w", encoding="utf-8") as f:
    f.write("\n".join(lines))
print("Created business_intelligence_report.json and business_intelligence_report.md")

## Next steps
- Add business-specific KPI definitions and context.
- Add schema validation, anomaly detection, and tests.
- Add optional LLM summarization only after configuring a model and validating output.
- Build a Streamlit/Gradio dashboard.
- Add access controls, audit logs, monitoring, and human review before using sensitive data.

**Limitations:** This starter uses deterministic Python functions coordinated as agents, not autonomous LLM agents. Demo data is synthetic. Validate calculations and data before using insights for real decisions.